# 04 - Tariff Table & Business Recommendation: Motorcycle Insurance
Load the saved frequency and severity models, build a full pricing tariff (base rate grid + modifier factors), and summarize the business recommendation.

### หน้านี้ทำไปเพื่ออะไร (ภาษาไทย)

3 notebook ก่อนหน้า (cleaning → EDA → GLM model) เป็นขั้นตอน "หาความรู้/สร้างโมเดล" — แต่โมเดลอย่างเดียวยังใช้ตั้งราคาขายจริงไม่ได้ทันที เพราะมันตอบได้แค่ทีละกรมธรรม์ ไม่ได้เป็นตารางราคาที่พร้อมใช้งาน

**หน้านี้ทำ 3 อย่าง:**
1. **สร้างตารางราคา (tariff table)** — เอาโมเดลที่ fit ไว้แล้ว (โหลดจากไฟล์ ไม่ต้อง fit ใหม่) มาคำนวณราคาให้ครบทุกกลุ่มลูกค้าที่เป็นไปได้ ไม่ใช่แค่ตัวอย่างเดียว
2. **ตรวจสอบว่าตารางราคาถูกต้อง** — เทียบว่า "ราคาจากตาราง" กับ "ราคาที่โมเดลทำนายตรง ๆ" ให้ค่าเท่ากันจริงไหม
3. **สรุปเป็นข้อเสนอแนะเชิงธุรกิจ (business recommendation)** — แปลตัวเลขทั้งหมดให้เป็นภาษาที่คนไม่ใช่สายเทคนิคก็เข้าใจได้ เช่น "กลุ่มไหนควรขึ้นราคา" "ปัจจัยไหนไม่ควรใช้ตั้งราคาแล้ว"

**ประโยชน์ที่ได้:** จากการดูหน้านี้ เราจะได้ **ตารางราคาที่ใช้งานได้จริง** + **ข้อเสนอแนะที่พร้อมเอาไปคุยกับผู้บริหาร** ไม่ใช่แค่ตัวเลขทางสถิติเฉย ๆ — นี่คือ output สุดท้ายของทั้งโปรเจกต์

In [1]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

MODELS_DIR = Path("../models")
freq_model = sm.load(MODELS_DIR / "frequency_glm.pickle")
sev_model = sm.load(MODELS_DIR / "severity_glm.pickle")

df = pd.read_csv("../data/processed/motorcycle_clean.csv")
df_exp = df[~df["is_zero_exposure"]].copy()

age_bins = [0, 25, 35, 45, 55, 65, 100]
age_labels = ["<25", "25-34", "35-44", "45-54", "55-64", "65+"]
df_exp["age_group"] = pd.cut(df_exp["owner_age"], bins=age_bins, labels=age_labels, right=False)

veh_bins = [0, 3, 6, 10, 15, 100]
veh_labels = ["0-2", "3-5", "6-9", "10-14", "15+"]
df_exp["vehicle_age_group"] = pd.cut(df_exp["vehicle_age"], bins=veh_bins, labels=veh_labels, right=False).astype(str)

for col in ["gender", "zone", "vehicle_class", "bonus_class", "age_group"]:
    df_exp[col] = df_exp[col].astype(str)

print("Models loaded from disk (not refit).")
print(f"Frequency model params: {len(freq_model.params)} | Severity model params: {len(sev_model.params)}")

Models loaded from disk (not refit).
Frequency model params: 29 | Severity model params: 29


## 1. Build the base tariff grid

A real motor tariff is built as **a base rate grid for the strongest factors, plus modifier multipliers for the rest** — not one giant table with every factor crossed together (that would be 6 x 7 x 7 x 7 x 2 x 5 = 20,580 rows, unusable).

- **Base grid:** `age_group x zone x vehicle_class` (the 3 strongest, most reliable factors from notebook 03), priced at a reference profile: `bonus_class = 1`, `gender = Female`, `vehicle_age_group = 0-2` (a brand-new rider/bike/no-claims baseline).
- **Modifiers:** relativities for `bonus_class`, `gender`, `vehicle_age_group`, applied on top of the base rate.

Because neither GLM has interaction terms, this decomposition is exact — `base rate x modifiers` reproduces exactly what the full model would predict for any combination.

In [2]:
base_bonus = "1"
base_gender = "Female"
base_vehicle_age_group = "0-2"

zones = [str(z) for z in range(1, 8)]
vehicle_classes = [str(c) for c in range(1, 8)]

grid_rows = list(itertools.product(age_labels, zones, vehicle_classes))
tariff = pd.DataFrame(grid_rows, columns=["age_group", "zone", "vehicle_class"])
tariff["bonus_class"] = base_bonus
tariff["gender"] = base_gender
tariff["vehicle_age_group"] = base_vehicle_age_group

tariff["predicted_frequency"] = freq_model.predict(tariff, offset=np.zeros(len(tariff)))
tariff["predicted_severity"] = sev_model.predict(tariff)
tariff["base_pure_premium"] = tariff["predicted_frequency"] * tariff["predicted_severity"]

print(f"Base tariff grid: {len(tariff)} combinations (age_group x zone x vehicle_class)")
print(f"Reference profile held fixed: bonus_class={base_bonus}, gender={base_gender}, vehicle_age_group={base_vehicle_age_group}")
print("\nTop 10 highest base rates:")
tariff.sort_values("base_pure_premium", ascending=False).head(10)

Base tariff grid: 294 combinations (age_group x zone x vehicle_class)
Reference profile held fixed: bonus_class=1, gender=Female, vehicle_age_group=0-2

Top 10 highest base rates:


,age_group,zone,vehicle_class,bonus_class,gender,vehicle_age_group,predicted_frequency,predicted_severity,base_pure_premium
5,<25,1,6,1,Female,0-2,0.264764,36895.876442,9768.686123
54,25-34,1,6,1,Female,0-2,0.136306,64775.202230,8829.233418
12,<25,2,6,1,Female,0-2,0.155056,43344.418516,6720.790537
61,25-34,2,6,1,Female,0-2,0.079826,76096.402788,6074.453377
6,<25,1,7,1,Female,0-2,0.167783,30484.879176,5114.834914
4,<25,1,5,1,Female,0-2,0.168087,28701.407460,4824.342753
1,<25,1,2,1,Female,0-2,0.170995,28038.575432,4794.455232
55,25-34,1,7,1,Female,0-2,0.086378,53519.916154,4622.942203
53,25-34,1,5,1,Female,0-2,0.086535,50388.814464,4360.386619
50,25-34,1,2,1,Female,0-2,0.088032,49225.132155,4333.373376


## 2. Build the modifier tables

For `bonus_class`, `gender`, and `vehicle_age_group`, compute the combined (frequency x severity) relativity for every level relative to the reference profile above. The reference level itself gets a modifier of exactly 1.0 (no change).

In [3]:
def build_modifier_table(factor_col, levels, ref_level):
    rows = pd.DataFrame(
        {
            "age_group": "25-34",
            "zone": "1",
            "vehicle_class": "1",
            "bonus_class": base_bonus,
            "gender": base_gender,
            "vehicle_age_group": base_vehicle_age_group,
        },
        index=levels,
    )
    rows[factor_col] = levels

    freq = freq_model.predict(rows, offset=np.zeros(len(rows)))
    sev = sev_model.predict(rows)
    pure_premium = freq * sev

    return pd.DataFrame(
        {
            "predicted_frequency": freq,
            "predicted_severity": sev,
            "pure_premium": pure_premium,
            "modifier": pure_premium / pure_premium.loc[ref_level],
        }
    )


bonus_table = build_modifier_table("bonus_class", [str(i) for i in range(1, 8)], base_bonus)
gender_table = build_modifier_table("gender", ["Female", "Male"], base_gender)
vehicle_age_table = build_modifier_table("vehicle_age_group", veh_labels, base_vehicle_age_group)

print("Bonus class modifier (relative to bonus_class = 1):")
print(bonus_table["modifier"])
print("\nGender modifier (relative to Female):")
print(gender_table["modifier"])
print("\nVehicle age modifier (relative to 0-2 years):")
print(vehicle_age_table["modifier"])

Bonus class modifier (relative to bonus_class = 1):
1    1.000000
2    1.103714
3    1.501502
4    1.167907
5    1.268017
6    1.714920
7    1.176810
Name: modifier, dtype: float64

Gender modifier (relative to Female):
Female    1.000000
Male      1.409258
Name: modifier, dtype: float64

Vehicle age modifier (relative to 0-2 years):
0-2      1.000000
3-5      0.443604
6-9      0.271052
10-14    0.094856
15+      0.052424
Name: modifier, dtype: float64


## 3. Verify the decomposition is exact

Before trusting `base rate x modifiers`, check it against calling both models directly on a few full feature combinations that were **not** part of the reference profile.

In [4]:
def quote(age_group, zone, vehicle_class, bonus_class, gender, vehicle_age_group):
    row = pd.DataFrame(
        [{
            "age_group": age_group,
            "zone": zone,
            "vehicle_class": vehicle_class,
            "bonus_class": bonus_class,
            "gender": gender,
            "vehicle_age_group": vehicle_age_group,
        }]
    )
    freq = freq_model.predict(row, offset=np.zeros(len(row))).iloc[0]
    sev = sev_model.predict(row).iloc[0]
    return freq * sev


def quote_from_table(age_group, zone, vehicle_class, bonus_class, gender, vehicle_age_group):
    base_row = tariff[(tariff["age_group"] == age_group) & (tariff["zone"] == zone) & (tariff["vehicle_class"] == vehicle_class)]
    base_rate = base_row["base_pure_premium"].iloc[0]
    return base_rate * bonus_table.loc[bonus_class, "modifier"] * gender_table.loc[gender, "modifier"] * vehicle_age_table.loc[vehicle_age_group, "modifier"]


# spot-check three combinations that are NOT the reference profile
test_cases = [
    ("<25", "2", "6", "7", "Male", "15+"),
    ("45-54", "3", "2", "4", "Female", "6-9"),
    ("65+", "1", "6", "1", "Male", "0-2"),
]

for case in test_cases:
    direct = quote(*case)
    from_table = quote_from_table(*case)
    print(f"{case}: direct model = {direct:,.1f} | base x modifiers = {from_table:,.1f} | match: {np.isclose(direct, from_table)}")

('<25', '2', '6', '7', 'Male', '15+'): direct model = 584.3 | base x modifiers = 584.3 | match: True
('45-54', '3', '2', '4', 'Female', '6-9'): direct model = 68.5 | base x modifiers = 68.5 | match: True
('65+', '1', '6', '1', 'Male', '0-2'): direct model = 874.2 | base x modifiers = 874.2 | match: True


## 4. Lowest-risk vs highest-risk example quotes

Put the whole pricing model to work: compare the cheapest and most expensive realistic policyholder profiles.

In [6]:
cheapest_base = tariff.sort_values("base_pure_premium").iloc[0]
priciest_base = tariff.sort_values("base_pure_premium", ascending=False).iloc[0]

cheapest_modifiers = (
    bonus_table["modifier"].min(),
    gender_table["modifier"].min(),
    vehicle_age_table["modifier"].min(),
)
priciest_modifiers = (
    bonus_table["modifier"].max(),
    gender_table["modifier"].max(),
    vehicle_age_table["modifier"].max(),
)

cheapest_quote = cheapest_base["base_pure_premium"] * np.prod(cheapest_modifiers)
priciest_quote = priciest_base["base_pure_premium"] * np.prod(priciest_modifiers)

print("CHEAPEST realistic policy:")
print(f"  age {cheapest_base['age_group']}, zone {cheapest_base['zone']}, vehicle_class {cheapest_base['vehicle_class']}, "
      f"best bonus/gender/vehicle-age modifiers")
print(f"  -> predicted pure premium: {cheapest_quote:,.4f} per policy-year\n")

print("MOST EXPENSIVE realistic policy:")
print(f"  age {priciest_base['age_group']}, zone {priciest_base['zone']}, vehicle_class {priciest_base['vehicle_class']}, "
      f"worst bonus/gender/vehicle-age modifiers")
print(f"  -> predicted pure premium: {priciest_quote:,.1f} per policy-year\n")

print(f"Price ratio: the most expensive policy costs {priciest_quote / cheapest_quote:,.0f}x the cheapest.")

CHEAPEST realistic policy:
  age 65+, zone 7, vehicle_class 4, best bonus/gender/vehicle-age modifiers
  -> predicted pure premium: 0.0290 per policy-year

MOST EXPENSIVE realistic policy:
  age <25, zone 1, vehicle_class 6, worst bonus/gender/vehicle-age modifiers
  -> predicted pure premium: 23,608.6 per policy-year

Price ratio: the most expensive policy costs 814,415x the cheapest.


**Caution — that 814,415x ratio is a modeling artifact, not a real price gap.** The "cheapest" combination stacks together the most extreme end of *five* multipliers at once, including `zone 7` — the same zone flagged in notebook 03 as having only **1 claim on record**, so its severity estimate is essentially a guess. Multiplying several uncertain extremes together compounds the uncertainty into something meaningless. In a real pricing system, an actuary would cap or smooth rates for these tiny, poorly-observed segments rather than quote them directly.

A more useful, realistic comparison uses two profiles that actually have decent data behind them:

In [7]:
higher_risk_profile = ("<25", "2", "6", "1", "Male", "0-2")  # young rider, risky zone, high-power new bike
lower_risk_profile = ("45-54", "4", "1", "7", "Female", "15+")  # mature rider, low-power old bike, max no-claims bonus

higher_risk_quote = quote(*higher_risk_profile)
lower_risk_quote = quote(*lower_risk_profile)

print(f"Higher-risk profile {higher_risk_profile}: {higher_risk_quote:,.0f} per policy-year")
print(f"Lower-risk profile  {lower_risk_profile}: {lower_risk_quote:,.0f} per policy-year")
print(f"Realistic price ratio: {higher_risk_quote / lower_risk_quote:,.0f}x")

Higher-risk profile ('<25', '2', '6', '1', 'Male', '0-2'): 9,471 per policy-year
Lower-risk profile  ('45-54', '4', '1', '7', 'Female', '15+'): 5 per policy-year
Realistic price ratio: 1,859x


Even using two individually well-observed profiles, the ratio (1,859x) is still far more extreme than what real insurers actually charge between their best and worst customers (typically single/low-double-digit multiples). This is a known characteristic of **multiplicative rating with no interaction terms and no caps**: each factor (age, zone, vehicle class, bonus, gender, vehicle age) looks reasonable on its own, but stacking several "cheap" or several "expensive" levels at once compounds into an extreme that the additive log-scale model was never really tested against. This is exactly why real tariff systems apply a **maximum combined discount/loading cap** (e.g. "no combination of factors can move the price by more than X%") rather than trusting the raw multiplicative product at the extremes. For the actual, real, data-backed headline number, the EDA's directly observed riskiest segment — **17.7x the overall average** for age&lt;25/zone2/vehicle_class6 policies with 75 real policy-years behind it — is the safer number to quote to a business stakeholder.

## 5. Save the tariff outputs

In [8]:
PROCESSED_DIR = Path("../data/processed")

tariff.to_csv(PROCESSED_DIR / "tariff_base_rates.csv", index=False)
bonus_table.to_csv(PROCESSED_DIR / "tariff_modifier_bonus_class.csv")
gender_table.to_csv(PROCESSED_DIR / "tariff_modifier_gender.csv")
vehicle_age_table.to_csv(PROCESSED_DIR / "tariff_modifier_vehicle_age.csv")

print("Saved tariff outputs to data/processed/:")
for name in ["tariff_base_rates.csv", "tariff_modifier_bonus_class.csv", "tariff_modifier_gender.csv", "tariff_modifier_vehicle_age.csv"]:
    print(f"  - {name}")

Saved tariff outputs to data/processed/:
  - tariff_base_rates.csv
  - tariff_modifier_bonus_class.csv
  - tariff_modifier_gender.csv
  - tariff_modifier_vehicle_age.csv


## Business recommendation

**What we built:** a two-part pricing model (Poisson frequency + Gamma severity GLM, validated to within 3.1% of actual total cost) turned into a usable tariff — a 294-row base rate grid (`age_group x zone x vehicle_class`) plus three modifier tables (`bonus_class`, `gender`, `vehicle_age_group`).

**Recommendation 1 — reprice the confirmed high-risk segment.** Riders under 25, in zone 2, on vehicle class 6 bikes are a real, well-observed risk group (75 policy-years, 8 claims): they cost **~17.7x** the average policy but the base tariff before modifiers only reflects part of that gap. Current flat/under-differentiated pricing for this group is very likely under-priced — recommend a targeted rate increase here, backed by real claims history, not just the model's smoothed estimate.

**Recommendation 2 — stop relying on `bonus_class` as a standalone rating signal.** It was not statistically significant in either model once age, zone, and vehicle class were controlled for. The no-claims discount can stay as a customer-retention/loyalty tool, but it should not be treated as an accurate risk indicator on its own — age and vehicle class already capture most of the real risk signal that bonus class was assumed to represent.

**Recommendation 3 — cap combined discounts/loadings.** The tariff's multiplicative structure (no interaction terms) can compound multiple factors into unrealistic extremes (>1,000x) when several "best" or "worst" levels stack together. Before this goes into production, add a sensible cap (e.g. a maximum combined multiplier) — a standard practice in real rating engines.

**Recommendation 4 — treat zone 7 and other thin segments with caution.** Zone 7 has exactly 1 claim on record; its severity estimate should not be trusted for pricing until more data accumulates. Consider merging it with a similar low-population zone or applying credibility weighting toward the overall average until it has more claims history.

**What's in the repo now:**
- `notebooks/01_data_cleaning.ipynb` — renamed columns, data quality checks
- `notebooks/02_eda.ipynb` — frequency/severity by segment, key patterns
- `notebooks/03_frequency_severity_glm.ipynb` — Poisson + Gamma GLMs, validated, saved to `models/`
- `notebooks/04_tariff_and_recommendation.ipynb` — full tariff table + business recommendation (this notebook)
- `data/processed/tariff_base_rates.csv` and 3 modifier CSVs — the actual pricing output
- `models/frequency_glm.pickle`, `models/severity_glm.pickle` — the fitted models, reusable without refitting